In [1]:
# 1 - CREATE - 2_QEMU_XV6_RUN.txt - START

# TIME START - BEGIN
import time
TIME_START = time.time()
# TIME START - FINISH

# IMPORT - BEGIN
from pwn import *
context.log_level = 'critical'
import re, subprocess, sys
# IMPORT - FINISH

# PATH - BEGIN
FILE_REGISTER_DEFAULT = 'TEXT/QEMU_XV6_REGISTER_DEFAULT.txt'
FILE_REGISTER_CHANGE = 'TEXT/QEMU_XV6_REGISTER_CHANGE.txt'
FILE_BOOTLOADER_CODE = 'bootloader/code.S'
FILE_INPUT = 'TEXT/1_QEMU_XV6_INPUT.txt'
FILE_OUTPUT = 'TEXT/2_QEMU_XV6_RUN.txt'
# PATH - FINISH

# FUNCTION - BEGIN
def FUNCTION_PARSE_REGISTER(FILE_PATH):
    MAP_REGISTER = {}
    with open(FILE_PATH, 'r', encoding='utf-8') as FILE_HANDLE:
        for EACH_LINE in FILE_HANDLE:
            LIST_PART = EACH_LINE.split()
            if len(LIST_PART) >= SKIPPING_LINE_COUNT:
                MAP_REGISTER[LIST_PART[0]] = LIST_PART[1]
    return MAP_REGISTER

def FUNCTION_CLEAN_REGISTER_OUTPUT(STRING_OUTPUT):
    LIST_LINE = STRING_OUTPUT.splitlines()
    STRING_TRIMMED = '\n'.join(LIST_LINE[SKIPPING_LINE_COUNT:-1])
    return REGEX_ANSI_ESCAPE.sub('', STRING_TRIMMED)

def FUNCTION_HANDLE_QEMU_ERROR(LIST_CMDLINE):
    OBJECT_ERROR = subprocess.run(LIST_CMDLINE, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    print(OBJECT_ERROR.stderr.strip())

def FUNCTION_GET_CHANGED_REGISTER_INFO(OBJECT_IO, LIST_CHANGED_REGISTER):
    STRING_REGISTER = ' '.join(LIST_CHANGED_REGISTER)
    OBJECT_IO.sendline(f'info register {STRING_REGISTER}'.encode())
    STRING_OUTPUT = OBJECT_IO.clean().decode()
    LIST_LINE = STRING_OUTPUT.splitlines()
    if LIST_LINE and '(gdb)' in LIST_LINE[-1]:
        LIST_LINE.pop()
    return '\n'.join(LIST_LINE).strip()

def FUNCTION_TEST_TEST_CASE(OBJECT_IO):
    MAP_REGISTER_OLD = FUNCTION_PARSE_REGISTER(FILE_REGISTER_DEFAULT)
    MAP_REGISTER_NEW = FUNCTION_PARSE_REGISTER(FILE_REGISTER_CHANGE)
    if ONLY_SHOW_CHANGED_REGISTER:
        LIST_CHANGED_REGISTER = []
        SET_REGISTER_EXCLUDED = {'sp', 'fp', 'pc', 'mcycle', 'minstret', 'cycle', 'time', 'instret'}
        for EACH_REGISTER, EACH_VALUE_NEW in MAP_REGISTER_NEW.items():
            VALUE_OLD = MAP_REGISTER_OLD.get(EACH_REGISTER)
            if VALUE_OLD and EACH_REGISTER not in SET_REGISTER_EXCLUDED and EACH_VALUE_NEW != VALUE_OLD:
                LIST_CHANGED_REGISTER.append(EACH_REGISTER)
    else:
        LIST_CHANGED_REGISTER = LIST_REGISTER_CHOSEN
    if not LIST_CHANGED_REGISTER:
        STRING_OUTPUT_CLEAN = 'No registers changed!'
    else:
        STRING_OUTPUT_CLEAN = FUNCTION_GET_CHANGED_REGISTER_INFO(OBJECT_IO, LIST_CHANGED_REGISTER)
    print(STRING_OUTPUT_CLEAN)

def FUNCTION_EDIT_MAIN_S(STRING_CONTENT):
    with open(FILE_BOOTLOADER_CODE, 'w', encoding='utf-8') as FILE_HANDLE:
        FILE_HANDLE.write(STRING_CONTENT)

def FUNCTION_TIDY_OUTPUT_FILE(FILE_NAME):
    with open(FILE_NAME, 'r', encoding='utf-8') as FILE_HANDLE:
        LIST_LINE = FILE_HANDLE.readlines()
    INDEX_START = 0
    while INDEX_START < len(LIST_LINE) and not (LIST_LINE[INDEX_START].startswith('============') or 'START ITER' in LIST_LINE[INDEX_START]):
        INDEX_START += 1
    LIST_KEEP = LIST_LINE[INDEX_START:]
    LIST_COLLAPSED = []
    VALUE_BLANK_STREAK = 0
    for EACH_LINE in LIST_KEEP:
        if EACH_LINE.strip() == '':
            VALUE_BLANK_STREAK += 1
            if VALUE_BLANK_STREAK <= 1:
                LIST_COLLAPSED.append(EACH_LINE)
        else:
            VALUE_BLANK_STREAK = 0
            LIST_COLLAPSED.append(EACH_LINE)
    while LIST_COLLAPSED and LIST_COLLAPSED[-1].strip() == '':
        LIST_COLLAPSED.pop()
    if LIST_COLLAPSED and LIST_COLLAPSED[-1].endswith('\n'):
        LIST_COLLAPSED[-1] = LIST_COLLAPSED[-1].rstrip('\n')
    with open(FILE_NAME, 'w', encoding='utf-8') as FILE_HANDLE:
        FILE_HANDLE.writelines(LIST_COLLAPSED)

def FUNCTION_RUN_QEMU(FUNCTION_CALLBACK, FILE_REGISTER, MODE_VALUE):
    LIST_CMDLINE = ['./run', str(MODE_VALUE)]
    OBJECT_QEMU_IO = process(LIST_CMDLINE)
    try:
        OBJECT_QEMU_IO.recvuntil(b'*** Now run')
        OBJECT_GDB_IO = process('riscv64-unknown-elf-gdb')
        OBJECT_GDB_IO.sendline(b'b *test_end')
        OBJECT_GDB_IO.sendline(b'b *trap_vector')
        OBJECT_GDB_IO.sendline(b'c')
        STRING_BREAK = OBJECT_GDB_IO.recvuntil(b'Breakpoint 1, ', timeout=TIMEOUT_SECOND).decode()
        if not STRING_BREAK:
            OBJECT_GDB_IO.recvuntil(b'Breakpoint 2, ', timeout=TIMEOUT_SECOND).decode()
        OBJECT_GDB_IO.sendline(b'info all-registers')
        STRING_OUTPUT = OBJECT_GDB_IO.clean().decode()
        STRING_REGISTER_CLEANED = FUNCTION_CLEAN_REGISTER_OUTPUT(STRING_OUTPUT)
        with open(FILE_REGISTER, 'w', encoding='utf-8') as FILE_HANDLE:
            FILE_HANDLE.write(STRING_REGISTER_CLEANED)
        if FUNCTION_CALLBACK:
            FUNCTION_CALLBACK(OBJECT_GDB_IO)
        if not STRING_OUTPUT:
            print('ERROR OCCURRED WHILE EXECUTING TEST CASE!')
        OBJECT_GDB_IO.kill()
    except EOFError:
        FUNCTION_HANDLE_QEMU_ERROR(LIST_CMDLINE)
    finally:
        OBJECT_QEMU_IO.kill()
# FUNCTION - FINISH

# VARIABLE - BEGIN
MODE_DEFAULT = 3
SKIPPING_LINE_COUNT = 3
TIMEOUT_SECOND = 1
STRING_DEFAULT_TEST_CASE = '''li x0, 0
li x1, 0
li x2, 0
li x3, 0
li x4, 0
li x5, 0
li x6, 0
li x7, 0
li x8, 0
li x9, 0
li x10, 0
li x11, 0
li x12, 0
li x13, 0
li x14, 0
li x15, 0
li x16, 0
li x17, 0
li x18, 0
li x19, 0
li x20, 0
li x21, 0
li x22, 0
li x23, 0
li x24, 0
li x25, 0
li x26, 0
li x27, 0
li x28, 0
li x29, 0
li x30, 0
li x31, 0
fmv.d.x f0, x0
fmv.d.x f1, x0
fmv.d.x f2, x0
fmv.d.x f3, x0
fmv.d.x f4, x0
fmv.d.x f5, x0
fmv.d.x f6, x0
fmv.d.x f7, x0
fmv.d.x f8, x0
fmv.d.x f9, x0
fmv.d.x f10, x0
fmv.d.x f11, x0
fmv.d.x f12, x0
fmv.d.x f13, x0
fmv.d.x f14, x0
fmv.d.x f15, x0
fmv.d.x f16, x0
fmv.d.x f17, x0
fmv.d.x f18, x0
fmv.d.x f19, x0
fmv.d.x f20, x0
fmv.d.x f21, x0
fmv.d.x f22, x0
fmv.d.x f23, x0
fmv.d.x f24, x0
fmv.d.x f25, x0
fmv.d.x f26, x0
fmv.d.x f27, x0
fmv.d.x f28, x0
fmv.d.x f29, x0
fmv.d.x f30, x0
fmv.d.x f31, x0
vsetvli x0, x0, e32, m1, ta, ma
vmv.v.i v0, 0
vmv.v.i v1, 0
vmv.v.i v2, 0
vmv.v.i v3, 0
vmv.v.i v4, 0
vmv.v.i v5, 0
vmv.v.i v6, 0
vmv.v.i v7, 0
vmv.v.i v8, 0
vmv.v.i v9, 0
vmv.v.i v10, 0
vmv.v.i v11, 0
vmv.v.i v12, 0
vmv.v.i v13, 0
vmv.v.i v14, 0
vmv.v.i v15, 0
vmv.v.i v16, 0
vmv.v.i v17, 0
vmv.v.i v18, 0
vmv.v.i v19, 0
vmv.v.i v20, 0
vmv.v.i v21, 0
vmv.v.i v22, 0
vmv.v.i v23, 0
vmv.v.i v24, 0
vmv.v.i v25, 0
vmv.v.i v26, 0
vmv.v.i v27, 0
vmv.v.i v28, 0
vmv.v.i v29, 0
vmv.v.i v30, 0
vmv.v.i v31, 0

'''
REGEX_ANSI_ESCAPE = re.compile(r'\x1B\[[0-?]*[ -/]*[@-~]')
ONLY_SHOW_CHANGED_REGISTER = False

# IMPORTANT - BEGIN
LIST_REGISTER_CHOSEN = ['priv', 'pc', 'a0', 'a1', 'a2', 'a3', 't0', 't1', 't2', 't3', 'f0', 'f1', 'f2', 'f3', 'ft0', 'ft1', 'ft2', 'ft3', 'v0', 'v1', 'v2', 'v3', 'x1', 'x2', 'x3', 'x4', 'x5', 'x6', 'x7', 'x8', 'x9', 'x10', 'x11', 'x12', 'x13', 'x14', 'x15', 'x16', 'x17', 'x18', 'x19', 'x20', 'x21', 'x22', 'x23', 'x24', 'x25', 'x26', 'x27', 'x28', 'x29', 'x30', 'x31', 'ra', 'sp', 'gp', 'tp', 'mstatus', 'mcause']
IS_OUTPUT = False
# IMPORTANT - FINISH

STRING_CODE_TEMPLATE = '''
.globl test_start

test_start:
{test_case}

test_end:
j test_end

.global trap_vector
.align 4

trap_vector:
j trap_vector
'''
# VARIABLE - FINISH

# CODE - BEGIN
if IS_OUTPUT:
    class CLASS_STDOUT_TO_FILE:
        def __init__(self, FILE_NAME):
            self.file = open(FILE_NAME, 'w', encoding='utf-8')
            self.stdout = sys.__stdout__
        def write(self, STRING_MESSAGE):
            self.file.write(REGEX_ANSI_ESCAPE.sub('', STRING_MESSAGE))
            self.stdout.write(STRING_MESSAGE)
        def flush(self):
            self.file.flush()
            self.stdout.flush()
        def close(self):
            self.file.close()
else:
    class CLASS_STDOUT_TO_FILE:
        def __init__(self, FILE_NAME):
            self.file = open(FILE_NAME, 'w', encoding='utf-8')
        def write(self, STRING_MESSAGE):
            self.file.write(REGEX_ANSI_ESCAPE.sub('', STRING_MESSAGE))
        def flush(self):
            self.file.flush()
        def close(self):
            self.file.close()

OBJECT_STDOUT_FILE = CLASS_STDOUT_TO_FILE(FILE_OUTPUT)
OBJECT_STDOUT_OLD = sys.stdout
sys.stdout = OBJECT_STDOUT_FILE

with open(FILE_INPUT, 'r', encoding='utf-8') as FILE_HANDLE:
    STRING_TEST_CASE = FILE_HANDLE.read()

LIST_BLOCK = [EACH_BLOCK.strip().split('\n') for EACH_BLOCK in STRING_TEST_CASE.strip().split('\n\n') if EACH_BLOCK.strip()]
LIST_TEST_CASE = [[EACH_LINE.strip() for EACH_LINE in EACH_BLOCK if EACH_LINE.strip()] for EACH_BLOCK in LIST_BLOCK]
STRING_LIST_TEST_CASE = f'LIST_TEST_CASE = {repr(LIST_TEST_CASE)}'

exec(STRING_LIST_TEST_CASE)

LIST_DEFAULT_TEST_CASE = [STRING_CODE_TEMPLATE.format(test_case=STRING_DEFAULT_TEST_CASE)]
LIST_CHANGED_TEST_CASE = [STRING_CODE_TEMPLATE.format(test_case='\n'.join(STRING_DEFAULT_TEST_CASE.splitlines() + EACH_TEST_CASE) if EACH_TEST_CASE else STRING_DEFAULT_TEST_CASE) for EACH_TEST_CASE in LIST_TEST_CASE]

try:
    FUNCTION_EDIT_MAIN_S(LIST_DEFAULT_TEST_CASE[0])
    FUNCTION_RUN_QEMU(None, FILE_REGISTER_DEFAULT, MODE_DEFAULT)
    for EACH_INDEX, (EACH_CONTENT_NEW, EACH_TEST_CASE) in enumerate(zip(LIST_CHANGED_TEST_CASE, LIST_TEST_CASE)):
        FUNCTION_EDIT_MAIN_S(EACH_CONTENT_NEW.strip())
        print(f'============\nSTART ITER {EACH_INDEX + 1}\n')
        print('\n'.join(EACH_TEST_CASE) + '\n')
        MODE_VALUE = MODE_DEFAULT
        FUNCTION_RUN_QEMU(FUNCTION_TEST_TEST_CASE, FILE_REGISTER_CHANGE, MODE_VALUE)
        print(f'\nEND ITER {EACH_INDEX + 1}\n==========\n')
finally:
    sys.stdout = OBJECT_STDOUT_OLD
    OBJECT_STDOUT_FILE.close()
    FUNCTION_TIDY_OUTPUT_FILE(FILE_OUTPUT)
# CODE - FINISH

# TIME END - BEGIN
TIME_END = time.time()
DURATION_IN_SECOND = TIME_END - TIME_START
DAYS = int(DURATION_IN_SECOND // 86400)
DURATION_IN_SECOND %= 86400
HOURS = int(DURATION_IN_SECOND // 3600)
DURATION_IN_SECOND %= 3600
MINUTES = int(DURATION_IN_SECOND // 60)
DURATION_IN_SECOND %= 60
PARTS = []
if DAYS:
    PARTS.append(f'{DAYS} day(s)')
if HOURS:
    PARTS.append(f'{HOURS} hour(s)')
if MINUTES:
    PARTS.append(f'{MINUTES} minute(s)')
if DURATION_IN_SECOND or not PARTS:
    PARTS.append(f'{DURATION_IN_SECOND:.2f} second(s)')
print(' '.join(PARTS))
# TIME END - FINISH

# DONE - BEGIN
print('====\nDONE\n====')
# DONE - FINISH

# 1 - CREATE - 2_QEMU_XV6_RUN.txt - END

1 minute(s) 13.88 second(s)
====
DONE
====


In [2]:
# 2 - CREATE - 3_QEMU_XV6_CHECK.txt - START

# TIME START - BEGIN
import time
TIME_START = time.time()
# TIME START - FINISH

# IMPORT - BEGIN
import operator, re
# IMPORT - FINISH

# PATH - BEGIN
FILE_INPUT = 'TEXT/2_QEMU_XV6_RUN.txt'
FILE_OUTPUT = 'TEXT/3_QEMU_XV6_CHECK.txt'
# PATH - FINISH

# FUNCTION - BEGIN
def FUNCTION_CANONICAL_REGISTER(STRING_NAME: str) -> str:
    STRING_KEY = STRING_NAME.strip()
    MATCH_V = re.fullmatch(r'(v\d+)\[(\d+)\]', STRING_KEY)
    if MATCH_V:
        return f'{MATCH_V.group(1)}.w[{MATCH_V.group(2)}]'
    MATCH_V2 = re.fullmatch(r'(v\d+)\.(b|s|w|l|q)\[(\d+)\]', STRING_KEY)
    if MATCH_V2:
        return f'{MATCH_V2.group(1)}.{MATCH_V2.group(2)}[{MATCH_V2.group(3)}]'
    return MAP_REGISTER_ALIAS.get(STRING_KEY, STRING_KEY)

def FUNCTION_PARSE_VECTOR_LIST(STRING_BODY: str, TOTAL_LEN: int):
    LIST_OUT = []
    TOKENS = [x.strip() for x in STRING_BODY.split(',') if x.strip()]
    for TOK in TOKENS:
        MREP = re.fullmatch(r'(0x[0-9a-fA-F]+)\s*<repeats\s+(\d+)\s+times>', TOK)
        if MREP:
            VAL = int(MREP.group(1), 16)
            CNT = int(MREP.group(2))
            LIST_OUT.extend([VAL] * CNT)
            continue
        MHEX = re.fullmatch(r'0x[0-9a-fA-F]+', TOK)
        if MHEX:
            LIST_OUT.append(int(TOK, 16))
            continue
        continue
    if len(LIST_OUT) < TOTAL_LEN:
        LIST_OUT.extend([0] * (TOTAL_LEN - len(LIST_OUT)))
    return LIST_OUT[:TOTAL_LEN]

def FUNCTION_PARSE_INT(STRING_VALUE: str) -> int:
    STRING_LOWER = STRING_VALUE.strip().lower()
    if STRING_LOWER.startswith(('+0x', '-0x', '0x')):
        return int(STRING_VALUE, 16)
    return int(STRING_VALUE, 10)

def FUNCTION_PARSE_ITERATION(STRING_CONTENT):
    REGEX_ITERATION = re.compile(r'START ITER (\d+)(.*?)END ITER \1', re.DOTALL)
    LIST_ITERATION = REGEX_ITERATION.findall(STRING_CONTENT)
    return LIST_ITERATION

def FUNCTION_HAS_ERROR(STRING_ITERATION: str) -> bool:
    return re.search(r'\bpriv\b', STRING_ITERATION) is None

def FUNCTION_PARSE_EXPECTED_RESULT(STRING_ITERATION: str):
    STRING_VALUE_PATTERN = r'[+-]?(?:0x[0-9a-fA-F]+|\d+)'
    STRING_REGISTER_PATTERN = r'(?:[A-Za-z_]\w+|v\d+(?:\.(?:b|s|w|l|q))?\[\d+\])'
    STRING_RHS_PATTERN = rf'(?:{STRING_VALUE_PATTERN}|{STRING_REGISTER_PATTERN})'
    REGEX_RESULT_LINE = re.compile(r'#\s*Result:\s*(.*)')
    REGEX_CONDITION = re.compile(
        rf'\s*({STRING_REGISTER_PATTERN})\s*(==|=|!=|<=|>=|<|>)\s*({STRING_RHS_PATTERN})\s*'
    )
    LIST_OR_GROUPS = []
    for EACH_LINE in STRING_ITERATION.splitlines():
        MATCH_RESULT = REGEX_RESULT_LINE.search(EACH_LINE)
        if not MATCH_RESULT:
            continue
        STRING_TAIL = MATCH_RESULT.group(1)
        LIST_OR_PART = [x.strip() for x in STRING_TAIL.split('|') if x.strip()]
        for EACH_OR in LIST_OR_PART:
            LIST_AND = []
            LIST_AND_PART = [x.strip() for x in EACH_OR.split('&') if x.strip()]
            for EACH_PART in LIST_AND_PART:
                MATCH_COND = REGEX_CONDITION.fullmatch(EACH_PART)
                if not MATCH_COND:
                    continue
                STRING_REG, STRING_OP, STRING_RHS_TOKEN = MATCH_COND.groups()
                STRING_REG = FUNCTION_CANONICAL_REGISTER(STRING_REG)
                if STRING_OP == '=':
                    STRING_OP = '=='
                LIST_AND.append((STRING_REG, STRING_OP, STRING_RHS_TOKEN))
            if LIST_AND:
                LIST_OR_GROUPS.append(LIST_AND)
    return LIST_OR_GROUPS

def FUNCTION_PARSE_ACTUAL_REGISTER(STRING_ITERATION: str):
    STRING_VALUE_PATTERN = r'[+-]?(?:0x[0-9a-fA-F]+|\d+)'
    REGEX_GPR = re.compile(rf'(\w+)\s+0x[0-9a-fA-F]+\s+({STRING_VALUE_PATTERN})')
    REGEX_FP_RAW = re.compile(
        r'^(f(?:t|s|a)?\d+)\s+[\s\S]*?\(raw\s+0x([0-9a-fA-F]+)\)',
        re.MULTILINE
    )
    REGEX_F_DOUBLE = re.compile(
        r'^(f\d+)\s+\{float\s*=\s*0x[0-9a-fA-F]+,\s*double\s*=\s*0x([0-9a-fA-F]+)\}',
        re.MULTILINE
    )
    MAP_ACTUAL = {}
    for EACH_REG, EACH_VALUE in REGEX_GPR.findall(STRING_ITERATION):
        try:
            VALUE_PARSED = FUNCTION_PARSE_INT(EACH_VALUE)
        except ValueError:
            continue
        STRING_CANON = FUNCTION_CANONICAL_REGISTER(EACH_REG)
        MAP_ACTUAL[STRING_CANON] = VALUE_PARSED
        MAP_ACTUAL[EACH_REG] = VALUE_PARSED
    for EACH_REG, EACH_HEX in REGEX_FP_RAW.findall(STRING_ITERATION):
        VALUE_PARSED = int(EACH_HEX, 16)
        STRING_CANON = FUNCTION_CANONICAL_REGISTER(EACH_REG)
        MAP_ACTUAL[STRING_CANON] = VALUE_PARSED
        MAP_ACTUAL[EACH_REG] = VALUE_PARSED
    for EACH_REG, EACH_HEX_DOUBLE in REGEX_F_DOUBLE.findall(STRING_ITERATION):
        VALUE_PARSED = int(EACH_HEX_DOUBLE, 16)
        STRING_CANON = FUNCTION_CANONICAL_REGISTER(EACH_REG)
        MAP_ACTUAL[STRING_CANON] = VALUE_PARSED
        MAP_ACTUAL[EACH_REG] = VALUE_PARSED
    REGEX_VBLOCK = re.compile(r'^(v\d+)\s+(.*?)(?=^v\d+\s+|\Z)', re.MULTILINE | re.DOTALL)
    REGEX_VIEW = re.compile(r'\b([qlwsb])\s*=\s*\{(.*?)\}', re.DOTALL)
    MAP_LEN = {'b': 16, 's': 8, 'w': 4, 'l': 2, 'q': 1}
    MAP_MASK = {'b': (1 << 8) - 1, 's': (1 << 16) - 1, 'w': (1 << 32) - 1, 'l': (1 << 64) - 1, 'q': MASK64}
    for VREG, VBLOCK in REGEX_VBLOCK.findall(STRING_ITERATION):
        VREG = VREG.strip()
        for VIEW, BODY in REGEX_VIEW.findall(VBLOCK):
            VIEW = VIEW.strip()
            if VIEW not in MAP_LEN:
                continue
            LIST_VAL = FUNCTION_PARSE_VECTOR_LIST(BODY, MAP_LEN[VIEW])
            for IDX, VAL in enumerate(LIST_VAL):
                KEY = f'{VREG}.{VIEW}[{IDX}]'
                MAP_ACTUAL[KEY] = VAL & MAP_MASK[VIEW]
    return MAP_ACTUAL

def FUNCTION_VECTOR_VIEW_BITS(STRING_REG_KEY: str):
    M = re.fullmatch(r'v\d+\.(b|s|w|l|q)\[\d+\]', STRING_REG_KEY)
    if not M:
        return None
    return {'b': 8, 's': 16, 'w': 32, 'l': 64, 'q': 64}[M.group(1)]

def FUNCTION_HAS_RESULT_ONLY_ERROR(STRING_ITERATION: str) -> bool:
    return REGEX_RESULT_ONLY_ERROR.search(STRING_ITERATION) is not None

def FUNCTION_U64(VALUE_X: int) -> int:
    return VALUE_X & MASK64

def FUNCTION_CHECK_ITERATION(LIST_ITERATION):
    STRING_OUTPUT = ''
    for EACH_ITER_NUMBER, EACH_ITER_CONTENT in LIST_ITERATION:
        VALUE_ITER_NUMBER = int(EACH_ITER_NUMBER)
        if FUNCTION_HAS_RESULT_ONLY_ERROR(EACH_ITER_CONTENT):
            if FUNCTION_HAS_ERROR(EACH_ITER_CONTENT):
                continue
            STRING_OUTPUT += f'START ITER {VALUE_ITER_NUMBER} is false\n'
            continue
        if FUNCTION_HAS_ERROR(EACH_ITER_CONTENT):
            STRING_OUTPUT += f'START ITER {VALUE_ITER_NUMBER} is error\n'
            continue
        LIST_EXPECTED_GROUP = FUNCTION_PARSE_EXPECTED_RESULT(EACH_ITER_CONTENT)
        MAP_ACTUAL = FUNCTION_PARSE_ACTUAL_REGISTER(EACH_ITER_CONTENT)
        FLAG_MATCH = False
        for EACH_GROUP in LIST_EXPECTED_GROUP:
            GROUP_OK = True
            for EACH_REG, EACH_OP, EACH_RHS_TOKEN in EACH_GROUP:
                STRING_REG_KEY = FUNCTION_CANONICAL_REGISTER(EACH_REG)
                VALUE_ACTUAL = MAP_ACTUAL.get(STRING_REG_KEY)
                if VALUE_ACTUAL is None:
                    GROUP_OK = False
                    break
                try:
                    VALUE_EXPECTED = FUNCTION_PARSE_INT(EACH_RHS_TOKEN)
                except ValueError:
                    STRING_RHS_KEY = FUNCTION_CANONICAL_REGISTER(EACH_RHS_TOKEN)
                    VALUE_RHS = MAP_ACTUAL.get(STRING_RHS_KEY)
                    if VALUE_RHS is None:
                        GROUP_OK = False
                        break
                    VALUE_EXPECTED = VALUE_RHS
                if EACH_OP in ('==', '!='):
                    WIDTH_BITS = None
                
                    # NEW: kalau RHS literal hex 32/64-bit, pakai itu untuk semua register (GPR/FPR juga boleh)
                    WIDTH_BITS = FUNCTION_HEX_WIDTH_BITS(EACH_RHS_TOKEN)
                
                    # vector view tetap override
                    V_BITS = FUNCTION_VECTOR_VIEW_BITS(STRING_REG_KEY)
                    if V_BITS is not None:
                        WIDTH_BITS = V_BITS
                
                    if WIDTH_BITS == 8:
                        VALUE_A = VALUE_ACTUAL & ((1 << 8) - 1)
                        VALUE_E = VALUE_EXPECTED & ((1 << 8) - 1)
                    elif WIDTH_BITS == 16:
                        VALUE_A = VALUE_ACTUAL & ((1 << 16) - 1)
                        VALUE_E = VALUE_EXPECTED & ((1 << 16) - 1)
                    elif WIDTH_BITS == 32:
                        VALUE_A = VALUE_ACTUAL & MASK32
                        VALUE_E = VALUE_EXPECTED & MASK32
                    else:
                        VALUE_A = FUNCTION_U64(VALUE_ACTUAL)
                        VALUE_E = FUNCTION_U64(VALUE_EXPECTED)
                
                    if not MAP_OPERATOR[EACH_OP](VALUE_A, VALUE_E):
                        GROUP_OK = False
                        break
            if GROUP_OK:
                FLAG_MATCH = True
                break
        if not FLAG_MATCH:
            STRING_OUTPUT += f'START ITER {VALUE_ITER_NUMBER} is false\n'
    STRING_OUTPUT_FINAL = STRING_OUTPUT.strip()
    with open(FILE_OUTPUT, 'w', encoding='utf-8') as FILE_HANDLE:
        FILE_HANDLE.write(STRING_OUTPUT_FINAL)

def FUNCTION_IS_FLOAT_REG(STRING_REG_CANON: str) -> bool:
    return re.fullmatch(r'f(?:[0-9]|[12][0-9]|3[01])', STRING_REG_CANON) is not None

def FUNCTION_HEX_WIDTH_BITS(STRING_TOKEN: str):
    S = STRING_TOKEN.strip().lower()
    if S.startswith(('+0x', '-0x')):
        S = S[1:]
    if not S.startswith('0x'):
        return None
    HEXDIG = S[2:]
    if 1 <= len(HEXDIG) <= 8:
        return 32
    if 9 <= len(HEXDIG) <= 16:
        return 64
    return None
# FUNCTION - FINISH

# VARIABLE - BEGIN
REGEX_RESULT_ONLY_ERROR = re.compile(r'^\s*#\s*Result\s*:\s*error\s*$', re.IGNORECASE | re.MULTILINE)
MAP_OPERATOR = {'=': operator.eq, '==': operator.eq, '!=': operator.ne, '<': operator.lt, '<=': operator.le, '>': operator.gt, '>=': operator.ge}
MAP_REGISTER_ALIAS = {'x0': 'x0', 'zero': 'x0', 'x1': 'x1', 'ra': 'x1', 'x2': 'x2', 'sp': 'x2', 'x3': 'x3', 'gp': 'x3', 'x4': 'x4', 'tp': 'x4', 'x5': 'x5', 't0': 'x5', 'x6': 'x6', 't1': 'x6', 'x7': 'x7', 't2': 'x7', 'x8': 'x8', 's0': 'x8', 'fp': 'x8', 'x9': 'x9', 's1': 'x9', 'x10': 'x10', 'a0': 'x10', 'x11': 'x11', 'a1': 'x11', 'x12': 'x12', 'a2': 'x12', 'x13': 'x13', 'a3': 'x13', 'x14': 'x14', 'a4': 'x14', 'x15': 'x15', 'a5': 'x15', 'x16': 'x16', 'a6': 'x16', 'x17': 'x17', 'a7': 'x17', 'x18': 'x18', 's2': 'x18', 'x19': 'x19', 's3': 'x19', 'x20': 'x20', 's4': 'x20', 'x21': 'x21', 's5': 'x21', 'x22': 'x22', 's6': 'x22', 'x23': 'x23', 's7': 'x23', 'x24': 'x24', 's8': 'x24', 'x25': 'x25', 's9': 'x25', 'x26': 'x26', 's10': 'x26', 'x27': 'x27', 's11': 'x27', 'x28': 'x28', 't3': 'x28', 'x29': 'x29', 't4': 'x29', 'x30': 'x30', 't5': 'x30', 'x31': 'x31', 't6': 'x31'}
MASK64 = (1 << 64) - 1
MASK32 = (1 << 32) - 1
MAP_REGISTER_ALIAS.update({**{f'f{i}': f'f{i}' for i in range(32)}, 'ft0': 'f0', 'ft1': 'f1', 'ft2': 'f2', 'ft3': 'f3', 'ft4': 'f4', 'ft5': 'f5', 'ft6': 'f6', 'ft7': 'f7', 'ft8': 'f28', 'ft9': 'f29', 'ft10': 'f30', 'ft11': 'f31', 'fs0': 'f8', 'fs1': 'f9', 'fs2': 'f18', 'fs3': 'f19', 'fs4': 'f20', 'fs5': 'f21', 'fs6': 'f22', 'fs7': 'f23', 'fs8': 'f24', 'fs9': 'f25', 'fs10': 'f26', 'fs11': 'f27', 'fa0': 'f10', 'fa1': 'f11', 'fa2': 'f12', 'fa3': 'f13', 'fa4': 'f14', 'fa5': 'f15', 'fa6': 'f16', 'fa7': 'f17'})
# VARIABLE - FINISH

# CODE - BEGIN
with open(FILE_INPUT, 'r', encoding='utf-8') as FILE_HANDLE:
    STRING_CONTENT = FILE_HANDLE.read()

LIST_ITERATION = FUNCTION_PARSE_ITERATION(STRING_CONTENT)
FUNCTION_CHECK_ITERATION(LIST_ITERATION)
# CODE - FINISH

# TIME END - BEGIN
TIME_END = time.time()
DURATION_IN_SECOND = TIME_END - TIME_START
DAYS = int(DURATION_IN_SECOND // 86400)
DURATION_IN_SECOND %= 86400
HOURS = int(DURATION_IN_SECOND // 3600)
DURATION_IN_SECOND %= 3600
MINUTES = int(DURATION_IN_SECOND // 60)
DURATION_IN_SECOND %= 60
PARTS = []
if DAYS:
    PARTS.append(f'{DAYS} day(s)')
if HOURS:
    PARTS.append(f'{HOURS} hour(s)')
if MINUTES:
    PARTS.append(f'{MINUTES} minute(s)')
if DURATION_IN_SECOND or not PARTS:
    PARTS.append(f'{DURATION_IN_SECOND:.2f} second(s)')
print(' '.join(PARTS))
# TIME END - FINISH

# DONE - BEGIN
print('====\nDONE\n====')
# DONE - FINISH

# ALARM PLAY - BEGIN
import warnings
warnings.filterwarnings('ignore')
import os
os.environ['PYGAME_HIDE_SUPPORT_PROMPT'] = '1'
import pygame
import time
pygame.mixer.init()
pygame.mixer.music.load('alarm.mp3')
for _ in range(3):
    pygame.mixer.music.play()
    while pygame.mixer.music.get_busy():
        time.sleep(0.01)
pygame.mixer.quit()
# ALARM PLAY - FINISH

# 2 - CREATE - 3_QEMU_XV6_CHECK.txt - END

0.08 second(s)
====
DONE
====
